# Grid Search - PINN Helmholtz 2D

Complementa al notebook principal. Probamos muchas combinaciones de hiperparametros para M2 y vemos cual da el menor error.

Usamos PyTorch (en vez de TF como en el principal) porque para esta red chica corre mas rapido: no hay overhead de los `GradientTape` anidados.

Variamos: learning rate, ancho de la red, optimizador y activacion. Cada corrida son 40000 iteraciones, asi que el grid completo se va a ~2 horas en CPU.

### Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd
import itertools
import time

In [ ]:
LAMBDA = 1.0
A1, A2 = 1.0, 4.0

### Problema y red

Resolvemos $\Delta u + \lambda u = q$ en $[-1, 1]^2$ con $u = 0$ en los bordes, $u_{\text{exacta}} = \sin(\pi x)\sin(4\pi y)$ y $q = (-\pi^2 - 16\pi^2 + \lambda)\, u_{\text{exacta}}$.

In [ ]:
def u_exacta(x):
    return np.sin(A1 * np.pi * x[:, 0:1]) * np.sin(A2 * np.pi * x[:, 1:2])


def forcing(x):
    factor = -(A1 * np.pi) ** 2 - (A2 * np.pi) ** 2 + LAMBDA
    return factor * u_exacta(x)


def muestrear_interior(n):
    return np.random.uniform(-1, 1, size=(n, 2))


def muestrear_borde(n):
    m = n // 4
    abajo  = np.column_stack([np.random.uniform(-1, 1, m), -np.ones(m)])
    arriba = np.column_stack([np.random.uniform(-1, 1, m),  np.ones(m)])
    izq    = np.column_stack([-np.ones(m), np.random.uniform(-1, 1, m)])
    der    = np.column_stack([ np.ones(m), np.random.uniform(-1, 1, m)])
    return np.vstack([abajo, arriba, izq, der])

In [ ]:
n_eval = 100
x_eje = np.linspace(-1, 1, n_eval)
y_eje = np.linspace(-1, 1, n_eval)
X1, X2 = np.meshgrid(x_eje, y_eje)
puntos_eval = np.column_stack([X1.flatten(), X2.flatten()])
u_real_2d = u_exacta(puntos_eval).reshape(n_eval, n_eval)

La red toma el nombre de la activacion como parametro. Como necesitamos $u_{xx}$ y $u_{yy}$, la activacion tiene que ser derivable dos veces (descartamos ReLU). Probamos `tanh` (la del paper), `sigmoid` y `swish`.

In [ ]:
ACTIVACIONES = {
    "tanh": torch.tanh,
    "sigmoid": torch.sigmoid,
    "swish": lambda x: x * torch.sigmoid(x),
}


class PINN(nn.Module):
    def __init__(self, capas, activacion="tanh"):
        super().__init__()
        self.activacion_fn = ACTIVACIONES[activacion]
        self.capas = nn.ModuleList(
            [nn.Linear(capas[i], capas[i + 1]) for i in range(len(capas) - 1)]
        )
        for capa in self.capas:
            nn.init.xavier_normal_(capa.weight)
            nn.init.zeros_(capa.bias)

    def forward(self, x):
        for capa in self.capas[:-1]:
            x = self.activacion_fn(capa(x))
        return self.capas[-1](x)

### Funciones auxiliares

In [ ]:
def derivada(salida, entrada):
    return torch.autograd.grad(
        salida, entrada,
        grad_outputs=torch.ones_like(salida),
        create_graph=True,
    )[0]


def calcular_residuo(red, puntos):
    x = puntos[:, 0:1].clone().detach().requires_grad_(True)
    y = puntos[:, 1:2].clone().detach().requires_grad_(True)
    u = red(torch.cat([x, y], dim=1))
    u_x = derivada(u, x)
    u_y = derivada(u, y)
    u_xx = derivada(u_x, x)
    u_yy = derivada(u_y, y)
    return u_xx + u_yy + LAMBDA * u

In [ ]:
def construir_optimizador(nombre, parametros, lr):
    if nombre == "SGD":
        return torch.optim.SGD(parametros, lr=lr)
    if nombre == "Momentum":
        return torch.optim.SGD(parametros, lr=lr, momentum=0.9)
    if nombre == "NAG":
        return torch.optim.SGD(parametros, lr=lr, momentum=0.9, nesterov=True)
    if nombre == "AdaGrad":
        return torch.optim.Adagrad(parametros, lr=lr)
    if nombre == "RMSprop":
        return torch.optim.RMSprop(parametros, lr=lr)
    if nombre == "Adam":
        return torch.optim.Adam(parametros, lr=lr)
    if nombre == "AdamW":
        return torch.optim.AdamW(parametros, lr=lr)
    raise ValueError(f"Optimizador desconocido: {nombre}")

In [ ]:
def entrenar_combinacion(lr, ancho, prof, optimizador_nombre, activacion,
                         n_iter=40000, batch=128, seed=0, verbose=False):
    np.random.seed(seed)
    torch.manual_seed(seed)

    capas = [2] + [ancho] * prof + [1]
    red = PINN(capas, activacion=activacion)

    opt = construir_optimizador(optimizador_nombre, red.parameters(), lr)
    factor_decay = 0.9 ** (1.0 / 1000)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=factor_decay)

    lambda_bc = 1.0
    t0 = time.time()

    for it in range(n_iter):
        pts_int = torch.tensor(muestrear_interior(batch), dtype=torch.float32)
        pts_bnd = torch.tensor(muestrear_borde(batch), dtype=torch.float32)
        q_obj = torch.tensor(forcing(pts_int.numpy()), dtype=torch.float32)

        r_pred = calcular_residuo(red, pts_int)
        L_res = ((r_pred - q_obj) ** 2).mean()
        L_bc = (red(pts_bnd) ** 2).mean()

        # Update de lambda_bc cada 10 iters (paper, Algoritmo 1).
        if it % 10 == 0 and it > 0:
            maxs_res, means_bc = [], []
            for capa in red.capas:
                g_r = torch.autograd.grad(L_res, capa.weight, retain_graph=True)[0]
                g_b = torch.autograd.grad(lambda_bc * L_bc, capa.weight, retain_graph=True)[0]
                maxs_res.append(g_r.abs().max())
                means_bc.append(g_b.abs().mean())
            lambda_sug = (torch.stack(maxs_res).max() /
                          torch.stack(means_bc).mean()).item()
            lambda_bc = 0.9 * lambda_bc + 0.1 * lambda_sug

        perdida = L_res + lambda_bc * L_bc
        opt.zero_grad()
        perdida.backward()
        opt.step()
        scheduler.step()

        if verbose and it % 5000 == 0:
            print(f"  it={it}  L_res={L_res.item():.4f}  L_bc={L_bc.item():.4f}  lambda={lambda_bc:.2f}")

    red.eval()
    with torch.no_grad():
        u_pred = red(torch.tensor(puntos_eval, dtype=torch.float32)).numpy().flatten()
    error_l2 = (np.linalg.norm(u_real_2d.flatten() - u_pred)
                / np.linalg.norm(u_real_2d.flatten()))

    return {
        "error_l2": error_l2,
        "lambda_bc_final": lambda_bc,
        "tiempo_seg": time.time() - t0,
    }


### Grid

2 lrs x 2 anchos x 7 optimizadores x 1 activacion. Cada combinacion 40000 iters (~3-4 min).

In [ ]:
lrs = [0.001, 0.0005]
anchos = [50, 100]
profundidades = [3]
optimizadores = ["SGD", "Momentum", "NAG", "AdaGrad", "RMSprop", "Adam", "AdamW"]
activaciones = ["tanh"]

combinaciones = list(itertools.product(lrs, anchos, profundidades, optimizadores, activaciones))
total = len(combinaciones)
print(f"Combinaciones: {total}")
print(f"Tiempo estimado: ~{total*3}-{total*4} minutos")

In [ ]:
resultados = []
t_global = time.time()

for idx, (lr, ancho, prof, opt_nombre, act) in enumerate(combinaciones, 1):
    elapsed = (time.time() - t_global) / 60
    eta = (elapsed / max(idx - 1, 1)) * (total - idx + 1) if idx > 1 else 0
    print(f"[{idx:>2}/{total}] lr={lr}  ancho={ancho}  opt={opt_nombre:<8}  "
          f"act={act}  | {elapsed:.1f} min  | ETA: {eta:.0f} min")

    info = entrenar_combinacion(lr, ancho, prof, opt_nombre, act, n_iter=40000)
    info.update({
        "lr": lr, "ancho": ancho, "prof": prof,
        "optimizador": opt_nombre, "activacion": act,
    })
    resultados.append(info)
    print(f"     error L2 = {info['error_l2']:.6f}  "
          f"lambda_bc = {info['lambda_bc_final']:.2f}  "
          f"tiempo = {info['tiempo_seg']:.0f}s")

print(f"\nGrid terminado en {(time.time() - t_global)/60:.1f} minutos.")

### Resultados

In [ ]:
df = pd.DataFrame(resultados)
df = df[["lr", "ancho", "prof", "optimizador", "activacion",
         "error_l2", "lambda_bc_final", "tiempo_seg"]]
df = df.sort_values("error_l2").reset_index(drop=True)
df

**Mejor combinacion**

In [ ]:
mejor = df.iloc[0]
print(f"lr            = {mejor['lr']}")
print(f"ancho         = {mejor['ancho']}")
print(f"profundidad   = {mejor['prof']}")
print(f"optimizador   = {mejor['optimizador']}")
print(f"activacion    = {mejor['activacion']}")
print(f"error L2      = {mejor['error_l2']:.6f}")
print(f"lambda_bc     = {mejor['lambda_bc_final']:.2f}")
print(f"tiempo (seg)  = {mejor['tiempo_seg']:.0f}")

**Top 10**

In [ ]:
df.head(10).style.format({
    "error_l2": "{:.6f}",
    "lambda_bc_final": "{:.2f}",
    "tiempo_seg": "{:.0f}",
}).background_gradient(subset=["error_l2"], cmap="RdYlGn_r")

**Por optimizador**

In [ ]:
por_opt = df.groupby("optimizador")["error_l2"].agg(["min", "mean", "max"]).sort_values("min")
print(por_opt)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(por_opt))
w = 0.35
ax.bar(x - w/2, por_opt["min"], w, label="min", color="steelblue")
ax.bar(x + w/2, por_opt["mean"], w, label="mean", color="lightcoral")
ax.set_xticks(x)
ax.set_xticklabels(por_opt.index)
ax.set_ylabel("error L2 relativo")
ax.set_yscale("log")
ax.set_title("Error L2 por optimizador")
ax.legend()
plt.tight_layout()
plt.show()

**Por activacion**

In [ ]:
por_act = df.groupby("activacion")["error_l2"].agg(["min", "mean", "max"]).sort_values("min")
print(por_act)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(por_act))
w = 0.35
ax.bar(x - w/2, por_act["min"], w, label="min", color="steelblue")
ax.bar(x + w/2, por_act["mean"], w, label="mean", color="lightcoral")
ax.set_xticks(x)
ax.set_xticklabels(por_act.index)
ax.set_ylabel("error L2 relativo")
ax.set_yscale("log")
ax.set_title("Error L2 por activacion")
ax.legend()
plt.tight_layout()
plt.show()

**Heatmap optimizador x activacion**

In [ ]:
pivot = df.pivot_table(values="error_l2", index="optimizador",
                       columns="activacion", aggfunc="min")
print(pivot)

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot.values, cmap="RdYlGn_r", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i, j]:.4f}",
                ha="center", va="center", color="black", fontsize=10)
ax.set_xlabel("activacion")
ax.set_ylabel("optimizador")
ax.set_title("Mejor error L2 por par")
fig.colorbar(im, ax=ax, label="error L2")
plt.tight_layout()
plt.show()

**Por ancho**

In [ ]:
por_ancho = df.groupby("ancho")["error_l2"].agg(["min", "mean", "max"])
print(por_ancho)

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(por_ancho))
w = 0.35
ax.bar(x - w/2, por_ancho["min"], w, label="min", color="steelblue")
ax.bar(x + w/2, por_ancho["mean"], w, label="mean", color="lightcoral")
ax.set_xticks(x)
ax.set_xticklabels([f"ancho={a}" for a in por_ancho.index])
ax.set_ylabel("error L2 relativo")
ax.set_yscale("log")
ax.set_title("Error L2 por tamano de red")
ax.legend()
plt.tight_layout()
plt.show()

**Por learning rate**

In [ ]:
por_lr = df.groupby("lr")["error_l2"].agg(["min", "mean", "max"])
print(por_lr)

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(por_lr))
w = 0.35
ax.bar(x - w/2, por_lr["min"], w, label="min", color="steelblue")
ax.bar(x + w/2, por_lr["mean"], w, label="mean", color="lightcoral")
ax.set_xticks(x)
ax.set_xticklabels([f"lr={lr_v}" for lr_v in por_lr.index])
ax.set_ylabel("error L2 relativo")
ax.set_yscale("log")
ax.set_title("Error L2 por learning rate")
ax.legend()
plt.tight_layout()
plt.show()

**Ranking completo**

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
etiquetas = [
    f"lr={r['lr']}\n{r['optimizador']}\n{r['activacion']}, w={r['ancho']}"
    for _, r in df.iterrows()
]
colores = ["seagreen" if i == 0 else "steelblue" for i in range(len(df))]
ax.bar(range(len(df)), df["error_l2"].values, color=colores)
ax.set_xticks(range(len(df)))
ax.set_xticklabels(etiquetas, rotation=90, fontsize=7)
ax.set_yscale("log")
ax.set_ylabel("error L2 relativo")
ax.set_title(f"{len(df)} combinaciones ordenadas por error")
plt.tight_layout()
plt.show()